## Cosmic Web - JAX - Fusion - SOLUTION

[JAX](https://docs.jax.dev/en/latest/) runs the same NumPy code under `jax.jit`: the *traced graph* model. JAX traces the function once per input shape and hands the trace to XLA, its compiler, which fuses neighbouring operations into one kernel where it can. In this notebook we read what the compiler decided: the fusions of one call, the memory it kept, and the cost of a new shape.

**SOLUTION**

### Table of Contents

1. [Setup](#1-setup)
2. [Stencil](#2-stencil)
3. [Timing](#3-timing)
4. [Fusion](#4-fusion)
5. [Static shapes](#5-static-shapes)
6. [Launch count](#6-launch-count)
7. [Scatter](#7-scatter)
8. [Limitations and recap](#8-limitations-and-recap)

---

### 1. Setup

By default JAX reserves most of the device memory at its first operation. The first line turns that off, so the profiler probe gets its own share.

---

**Quick Docs**

- `XLA_PYTHON_CLIENT_PREALLOCATE=false`: allocate on demand instead of reserving the device; set before `import jax`.
- `jnp.asarray(a)`: a JAX array on a CuPy array's buffer, without a copy or an owning reference: keep the CuPy array alive.

In [ ]:
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

from functools import partial

import numpy as np
import jax
import jax.numpy as jnp

import cosmic_core as cw

N, K = cw.setup('jax', f'jax {jax.__version__} on {jax.devices()}, x64 {jax.config.jax_enable_x64}')
q_dev = cw.by_size(lambda n: jnp.asarray(cw.field(n)))

### 2. Stencil

The stencil is the CuPy code with `cp` replaced by `jnp`. Every shape must be known when XLA compiles, the histogram's included: `K` is a static argument and `jnp.bincount` takes `length=K`. Why can `length` not be traced? The cell after the stencil tries it.

---

**Quick Docs**

- `partial(jax.jit, static_argnames=('K',))`: compile with `K` baked into the executable; a new value is a new compile.
- `jnp.bincount(x, length=K)`: counts as int32; `length` fixes the output size.

In [ ]:
def ecc_cells_jax(q):
    '''Births of the eight cells each voxel owns, as traced array operations.'''
    ex = jnp.minimum(q, jnp.roll(q, -1, 0))
    ey = jnp.minimum(q, jnp.roll(q, -1, 1))
    ez = jnp.minimum(q, jnp.roll(q, -1, 2))
    fxy = jnp.minimum(ex, jnp.roll(ex, -1, 1))
    fxz = jnp.minimum(ex, jnp.roll(ex, -1, 2))
    fyz = jnp.minimum(ey, jnp.roll(ey, -1, 2))
    c = jnp.minimum(fxy, jnp.roll(fxy, -1, 2))
    return q, (ex, ey, ez), (fxy, fxz, fyz), c


@partial(jax.jit, static_argnames=('K',))
def ecc_hist_jax(q, K):
    '''Signed histogram of births (int32, length K), one compiled executable.'''
    v, (ex, ey, ez), (fxy, fxz, fyz), c = ecc_cells_jax(q)
    bc = lambda a: jnp.bincount(a.ravel(), length=K)
    return bc(v) - bc(ex) - bc(ey) - bc(ez) + bc(fxy) + bc(fxz) + bc(fyz) - bc(c)

In [ ]:
try:
    jax.jit(lambda a: jnp.bincount(a.ravel()))(q_dev(N))
except Exception as e:
    print(f'jnp.bincount without length= under jit: {type(e).__name__}')
    print('  ' + str(e).splitlines()[0][:160])

- Without `length` the output size is `x.max() + 1`, data a traced value does not have. XLA needs every shape before it compiles.

### 3. Timing

The runner calls the compiled function on the prebuilt device array and waits for the result: JAX returns before the device finishes. Jitting happens once, outside the runner.

---

**Quick Docs**

- `jax.block_until_ready(x)`: wait for every array in `x`; required before stopping any timer.

In [ ]:
def run_jax(n):
    hist = jax.block_until_ready(ecc_hist_jax(q_dev(n), K=K))
    return np.asarray(hist, dtype=np.int64)

Now let's make sure it works.

In [ ]:
_ = cw.measure('jax', run_jax, N, prepare=q_dev, cold_note='incl. trace and XLA compile')

### 4. Fusion

Two views of one function:

- The *jaxpr* is the trace: every primitive the function recorded, in order.
- The compiled text is what XLA made of it: fusions, each one kernel. A *scatter* writes each element to the address an index array names. `bincount` is one, with the operations that feed it fused in.

How many fusions and scatters would you expect from eight bincounts?

---

**Quick Docs**

- `jax.make_jaxpr(fn, static_argnums=(1,))(q, K)`: the trace as a jaxpr; `fn.__wrapped__` is the Python function under the jit.
- `ecc_hist_jax.lower(q, K=K).compile()`: the executable; `.as_text()` is the compiled text (HLO), `.memory_analysis()` its memory plan.
- `cw.compile_report(stage, jaxpr, compiled)`: counts the jaxpr's lines and primitives and the executable's fusions, scatters and temporaries; prints both lines and attaches the counts to the stage file.

In [ ]:
jaxpr = jax.make_jaxpr(ecc_hist_jax.__wrapped__, static_argnums=(1,))(q_dev(N), K)
compiled = ecc_hist_jax.lower(q_dev(N), K=K).compile()
cw.compile_report('jax', jaxpr, compiled)

- The jaxpr is the NumPy code as primitives: each roll a pair of slices and a concatenate, each bincount a scatter-add.
- Far fewer fusions than jaxpr lines: the roll and minimum chains fused into the scatters' operands. The scatters stayed separate, one per bincount.
- The temporaries are a few full-size arrays: most intermediates never reach memory.

### 5. Static shapes

The executable is specialised to its input shapes, so each new $N$ compiles again. The chart splits the first call at every size into the compile and the run.

---

**Quick Docs**

- `jax.clear_caches()`: forget every compiled executable, so the next call compiles.
- `cw.save_sweep(stage, run, first_call=True, prepare=q_dev)`: the sweep with the first call at every size timed too, after `prepare(n)`.
- `cw.plot_compile_share(stage, title, compile_label)`: the first call at every size of the stage's sweep, split into compile and run.

In [ ]:
jax.clear_caches()
cw.save_sweep('jax', run_jax, first_call=True, prepare=q_dev)
cw.plot_compile_share('jax', 'first call per size: compile against execution', compile_label='trace + XLA compile (one-time)')

- Every size pays a compile on its first call. The compile does not shrink with the problem, so at the smallest size it is most of the first call.

### 6. Launch count

One executable launches several kernels. The probe of the CuPy notebook traces one call, with `--cuda-graph-trace=node` so that kernels inside a CUDA graph (a batch of launches replayed as one) are listed one by one. How do the launches compare with the fusion count of §4, and which kernels take the time?

In [ ]:
JAX_PROBE = dict(header='''
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
from functools import partial
import jax
import jax.numpy as jnp
''', state=[q_dev])
trace = cw.trace_call('jax', run_jax, N, [ecc_cells_jax, ecc_hist_jax], **JAX_PROBE)
scatter_s = lambda rows: sum(d for _, d, name, kind in rows if kind == 'kernel' and 'scatter' in name)
print(f"scatter kernels: {scatter_s(trace['rows']) * 1e3:.2f} ms of {trace['gpu_busy_s'] * 1e3:.2f} ms GPU busy")

- One kernel per fusion of the compiled text, the eight scatters among them. One memcpy, the histogram: the device-to-host reads inside `cp.bincount` are gone.
- The scatter kernels take almost the whole of the GPU's time.

### 7. Scatter

Why do the scatters dominate? A scatter-add is one atomic add per element into one of $K$ addresses, and adds to the same address queue up.

Reshape the births into slabs of $S$ voxels and count each slab into its own row of an $(M, K)$ histogram, so the adds spread over $MK$ addresses. The TODO writes `bc`. The decorator with `S` static and the runner are written.

---

**Quick Docs**

- `a.reshape(-1, S)`: $M = N^3 / S$ rows of $S$ births; a power of two divides every sweep size.
- `jax.vmap(f)(rows)`: `f` on every row, here `jnp.bincount(row, length=K)`.
- `jnp.sum(x, axis=0)`: the rows summed into one histogram.

In [ ]:
@partial(jax.jit, static_argnames=('K', 'S'))
def ecc_hist_jax_private(q, K, S=2048):
    '''Signed histogram of births (int32, length K): each slab of S voxels counts into its own row first.'''
    v, (ex, ey, ez), (fxy, fxz, fyz), c = ecc_cells_jax(q)
    bc = lambda a: jnp.sum(jax.vmap(lambda row: jnp.bincount(row, length=K))(a.reshape(-1, S)), axis=0)
    return bc(v) - bc(ex) - bc(ey) - bc(ez) + bc(fxy) + bc(fxz) + bc(fyz) - bc(c)


def run_jax_private(n):
    return np.asarray(jax.block_until_ready(ecc_hist_jax_private(q_dev(n), K=K)), dtype=np.int64)

Now let's make sure it works.

In [ ]:
_ = cw.measure('jax_private', run_jax_private, N, prepare=q_dev, cold_note='incl. trace and XLA compile')

trace_private = cw.trace_call('jax_private', run_jax_private, N, [ecc_cells_jax, ecc_hist_jax_private], **JAX_PROBE)
print(f"scatter kernels: {scatter_s(trace['rows']) * 1e3:.2f} ms before, {scatter_s(trace_private['rows']) * 1e3:.2f} ms after")
cw.save_sweep('jax_private', run_jax_private)

- The scatter kernels shrank to a fraction of their time: the same primitive, spread over many more addresses.
- The launches grew by the row sums. The scatters still carry most of the call.

Takeaways:

- XLA fused the roll and minimum chains into the scatters' operands and materialised only a few intermediates.
- A scatter is a fusion boundary, and its cost is the queue of atomic adds at each address. A histogram row per slab shortened the queues without leaving the array model.
- `length=` and `S` are static because every shape is fixed when the executable is built. A new $N$ is a new executable.

### 8. Limitations and recap

XLA decides the fusion. Only the function can change. Integers are int32 unless x64 is enabled, so the gate casts on the host.

Recap:

- The CuPy code ran under `jax.jit` with `cp` replaced by `jnp` and `K` static.
- The trace located the cost in the scatters and a reshape cut it.

Next: `93__cosmic_web__cccl__iterators.ipynb` describes the stencil to a library algorithm through iterators.